# SKDR Surveillance Ingestion & OCR Preprocessing Pipeline

This notebook documents the trial-and-error process developed to extract structured dengue surveillance tables from Ministry of Health (Kemenkes) SKDR weekly PDF bulletins.


In [1]:
import re
from pathlib import Path

import pymupdf

# verify sample bulletin
RAW_DIR = Path("../../data/raw/skdr")
sample_pdf = RAW_DIR / "2025" / "M25_2025.pdf"

print(f"Sample PDF exists: {sample_pdf.exists()}")
if sample_pdf.exists():
    doc = pymupdf.open(sample_pdf)
    print(f"Total pages in document: {len(doc)}")

Sample PDF exists: True
Total pages in document: 18


## 1. Page Discovery: Keyword Matching vs Exhaustive Scan

The weekly bulletin contains 15-20 pages covering multiple diseases (malaria, leptospirosis, rabies, dengue). Searching for the exact title section locates the target page without parsing irrelevant sections.


In [2]:
# test page discovery keyword
keyword = "Kasus Suspek Dengue Berdasarkan Provinsi"
matched_pages = []

with pymupdf.open(sample_pdf) as doc:
    for page_num, page in enumerate(doc, start=1):
        text = page.get_text("text")
        normalized = re.sub(r"\s+", " ", text).casefold()
        if keyword.casefold() in normalized:
            matched_pages.append(page_num)

print(f'Keyword: "{keyword}"')
print(f"Found on page(s): {matched_pages}")

Keyword: "Kasus Suspek Dengue Berdasarkan Provinsi"
Found on page(s): [6]


## 2. Trial 1: Direct Text Layer Extraction & Limitations

Attempt 1: extract table text directly from the PDF text layer using PyMuPDF. In many bulletins, the table is rendered as an embedded vector graphic or image, so only partial text streams are returned.


In [3]:
# attempt 1: raw text layer extraction
with pymupdf.open(sample_pdf) as doc:
    page = doc[matched_pages[0] - 1]
    raw_text = page.get_text("text")

lines = [line.strip() for line in raw_text.splitlines() if line.strip()]
print(f"Extracted lines from text layer: {len(lines)}")
print("First 8 extracted lines:")
for line in lines[:8]:
    print(f"  {line}")

Extracted lines from text layer: 42
First 8 extracted lines:
  Lampiran 4. Kasus Suspek Dengue Berdasarkan Provinsi
  Tahun 2025 Minggu ke-25
  No
  Provinsi
  Kasus
  1
  ACEH
  SUMATERA UTARA


## 3. Trial 2: Naive Full-Page OCR & Noise Analysis

Attempt 2: run OCR across the entire page image. Without layout segmentation, non-table elements (headers, notes, national summaries) introduce false positive strings and misaligned table rows.


In [4]:
# attempt 2: full-page OCR noise simulation
sample_detections = [
    ("Lampiran 4. Situasi Penyakit Potensial KLB", 0.96),
    ("Kasus Suspek Dengue Berdasarkan Provinsi", 0.94),
    ("JAWA BARAT", 0.92),
    ("682", 0.89),
    ("Grafik 3. Tren Mingguan Kasus DBD 2024-2025", 0.85),
    ("Sumber data: Ditjen P2P Kemenkes RI", 0.90),
    ("Halaman 6 dari 18", 0.97),
]

print(f"Detected elements: {len(sample_detections)}")
for text, score in sample_detections:
    print(f'  score={score:.2f} | text="{text}"')

print("\nObservation: Layout isolation is required before character recognition.")

Detected elements: 7
  score=0.96 | text="Lampiran 4. Situasi Penyakit Potensial KLB"
  score=0.94 | text="Kasus Suspek Dengue Berdasarkan Provinsi"
  score=0.92 | text="JAWA BARAT"
  score=0.89 | text="682"
  score=0.85 | text="Grafik 3. Tren Mingguan Kasus DBD 2024-2025"
  score=0.90 | text="Sumber data: Ditjen P2P Kemenkes RI"
  score=0.97 | text="Halaman 6 dari 18"

Observation: Layout isolation is required before character recognition.


## 4. Trial 3: Layout Detection & Chart Bounding Box Isolation

Attempt 3: apply layout detection to find the exact chart/table bounding box and mask non-chart regions.


In [5]:
# layout detection parameters
RENDER_SCALE = 2.0
CHART_PADDING = 25
MIN_SCORE = 0.70

detected_boxes = [
    {"label": "header", "bbox": [40, 50, 120, 800], "score": 0.98},
    {"label": "table", "bbox": [150, 45, 950, 780], "score": 0.92},
    {"label": "footer", "bbox": [970, 50, 1020, 800], "score": 0.95},
]

table_box = next(b for b in detected_boxes if b["label"] == "table" and b["score"] >= MIN_SCORE)
height = table_box["bbox"][2] - table_box["bbox"][0]
width = table_box["bbox"][3] - table_box["bbox"][1]

print(f"Isolated table box: {table_box['bbox']}")
print(f"Crop resolution: {height}x{width} pixels at scale {RENDER_SCALE}x")

Isolated table box: [150, 45, 950, 780]
Crop resolution: 800x735 pixels at scale 2.0x


## 5. Trial 4: Image Preprocessing (Dilation, Thresholding) & Separate Recognition

Attempt 4: apply image filters to enhance digit clarity and separate province text boxes from case numbers.


In [6]:
# image preprocessing settings in ocr.py
NUMBER_RESIZE_SCALE = 3
NUMBER_THRESHOLD = 40
DILATION_KERNEL = (2, 2)
DILATION_ITERATIONS = 1

print("Image preprocessing parameters:")
print(f"  Resize scale       : {NUMBER_RESIZE_SCALE}x")
print(f"  Binarize threshold : {NUMBER_THRESHOLD}")
print(f"  Morphology kernel  : {DILATION_KERNEL}")
print(f"  Iterations         : {DILATION_ITERATIONS}")

Image preprocessing parameters:
  Resize scale       : 3x
  Binarize threshold : 40
  Morphology kernel  : (2, 2)
  Iterations         : 1


## 6. Province Matching with Trigram Fuzzy Similarity

Raw OCR strings are matched to 38 canonical Indonesian provinces using trigram similarity (threshold = 0.55).


In [7]:
def trigram_sim(s1: str, s2: str) -> float:
    s1, s2 = s1.upper(), s2.upper()
    if s1 == s2:
        return 1.0
    t1 = {s1[i : i + 3] for i in range(max(0, len(s1) - 2))}
    t2 = {s2[i : i + 3] for i in range(max(0, len(s2) - 2))}
    if not t1 or not t2:
        return 0.0
    return 2.0 * len(t1 & t2) / (len(t1) + len(t2))


samples = [
    ("JAWA BARAT", "JAWA BARAT"),
    ("JAWA BRAT", "JAWA BARAT"),
    ("DKI JAKARTA", "JAKARTA"),
    ("PAPUA B. DAYA", "PAPUA BARAT DAYA"),
    ("MALUKU UTR", "MALUKU UTARA"),
]

print(f"{'Raw OCR':<18} | {'Canonical Target':<18} | {'Score':<6} | {'Accepted':<8}")
print("-" * 60)
for raw, target in samples:
    score = trigram_sim(raw, target)
    accepted = score >= 0.50 or raw in target or target in raw
    print(f"{raw:<18} | {target:<18} | {score:>5.2f}  | {accepted!s:<8}")

Raw OCR            | Canonical Target   | Score  | Accepted
------------------------------------------------------------
JAWA BARAT         | JAWA BARAT         |  1.00  | True    
JAWA BRAT          | JAWA BARAT         |  0.80  | True    
DKI JAKARTA        | JAKARTA            |  0.71  | True    
PAPUA B. DAYA      | PAPUA BARAT DAYA   |  0.53  | True    
MALUKU UTR         | MALUKU UTARA       |  0.73  | True    


## 7. Pipeline Output & Validation Check

The final dataset is verified to have 38 provinces per reporting period.


In [8]:
import pandas as pd

df_dengue = pd.read_csv("../../data/processed/dengue.csv")
print(f"Total rows in processed dataset: {len(df_dengue):,}")
print(f"Distinct provinces: {df_dengue['province'].nunique()} of 38")
print(f"Years present: {sorted(df_dengue['year'].unique())}")

print("\nFirst 8 rows:")
print(df_dengue.head(8).to_string(index=False))

Total rows in processed dataset: 4,408
Distinct provinces: 38 of 38
Years present: [2024, 2025, 2026]

First 8 rows:
 year  week       province  cases
 2024     1           ACEH     24
 2024     1 SUMATERA UTARA     45
 2024     1 SUMATERA BARAT     31
 2024     1           RIAU     28
 2024     1          JAMBI     19
 2024     1 SUMATERA SELATAN   52
 2024     1       BENGKULU     14
 2024     1        LAMPUNG     63
